# 1D CNN and MLP Model Benchmark (`model_cnn.ipynb`)

This notebook trains and benchmarks 1D CNN and MLP architectures for per-finger touch detection.

## 1. Imports & Hyperparameters Setup

In [1]:
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report

# Set random seeds
RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

# Hyperparameters
SEQ_LEN = 5          # 5 frame timesteps
FEATURE_DIM = 16     # 8 coordinates + 8 velocities per timestep
BATCH_SIZE = 32
LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-4
EPOCHS = 40

# Setup device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Load & Normalize Datasets (5 Timesteps x 16 Features)

In [2]:
def extract_5step_features_16dim(csv_path):
    df = pd.read_csv(csv_path)
    n_samples = len(df)
    
    X = np.zeros((n_samples, 5, 16), dtype=np.float32)
    
    for k in range(1, 6):
        coord_cols = [
            f"wrist{k}_x", f"wrist{k}_y",
            f"mcp{k}_x", f"mcp{k}_y",
            f"pip{k}_x", f"pip{k}_y",
            f"dip{k}_x", f"dip{k}_y"
        ]
        coords_vals = df[coord_cols].fillna(0.0).values.astype(np.float32)
        
        if k == 1:
            vel_vals = np.zeros((n_samples, 8), dtype=np.float32)
        else:
            v_idx = k - 1
            vel_cols = [
                f"wrist{v_idx}_vx", f"wrist{v_idx}_vy",
                f"mcp{v_idx}_vx", f"mcp{v_idx}_vy",
                f"pip{v_idx}_vx", f"pip{v_idx}_vy",
                f"dip{v_idx}_vx", f"dip{v_idx}_vy"
            ]
            vel_vals = df[vel_cols].fillna(0.0).values.astype(np.float32)
            
        step_vals = np.hstack([coords_vals, vel_vals])
        X[:, k - 1, :] = step_vals
        
    target_col = "touch_finger" if "touch_finger" in df.columns else "touch"
    y = df[target_col].astype(str).str.strip().str.lower().isin(["1", "true", "t", "yes", "y"]).values.astype(np.float32)
    y = y.reshape(-1, 1)
    
    return X, y

TRAIN_CSV = "./data/training_data.csv"
TEST_CSV = "./data/test_data.csv"

X_train_np, y_train_np = extract_5step_features_16dim(TRAIN_CSV)
X_test_np, y_test_np = extract_5step_features_16dim(TEST_CSV)

# Normalize using StandardScaler
scaler = StandardScaler()
N_tr, T, C = X_train_np.shape
N_te, _, _ = X_test_np.shape

X_train_flat = X_train_np.reshape(N_tr, -1)
X_test_flat = X_test_np.reshape(N_te, -1)

X_train_scaled = scaler.fit_transform(X_train_flat).reshape(N_tr, T, C)
X_test_scaled = scaler.transform(X_test_flat).reshape(N_te, T, C)

X_train_tensor = torch.from_numpy(X_train_scaled).type(torch.float32)
y_train_tensor = torch.from_numpy(y_train_np).type(torch.float32)
X_test_tensor = torch.from_numpy(X_test_scaled).type(torch.float32)
y_test_tensor = torch.from_numpy(y_test_np).type(torch.float32)

print(f"X_train shape (5 steps x 16 features): {X_train_tensor.shape}, y_train shape: {y_train_tensor.shape}")
print(f"X_test shape  (5 steps x 16 features): {X_test_tensor.shape},  y_test shape:  {y_test_tensor.shape}")

## 3. PyTorch Dataset and DataLoader

In [3]:
class VelocitySequenceDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = VelocitySequenceDataset(X_train_tensor, y_train_tensor)
test_dataset = VelocitySequenceDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

## 4. Building 1D CNN and MLP Model Architectures

In [4]:
class FingerTouchCNN1D(nn.Module):
    def __init__(self, in_channels=16, num_classes=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_channels=in_channels, out_channels=32, kernel_size=3, padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Conv1d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, num_classes)
        )
        
    def forward(self, x):
        x_conv = x.permute(0, 2, 1)
        return self.net(x_conv)

class FingerTouchMLP(nn.Module):
    def __init__(self, input_dim=80, num_classes=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(input_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, num_classes)
        )
        
    def forward(self, x):
        return self.net(x)

model_cnn = FingerTouchCNN1D(in_channels=16).to(device)
model_mlp = FingerTouchMLP(input_dim=80).to(device)
print("1D CNN Architecture:\n", model_cnn)
print("\nMLP Architecture:\n", model_mlp)

## 5. Training Loop Helper

In [5]:
def train_and_evaluate(model, train_loader, test_loader, epochs=40, lr=0.001, weight_decay=1e-4):
    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
    
    def accuracy_fn(y_true, y_pred):
        return (torch.eq(y_true, y_pred).sum().item() / len(y_pred)) * 100.0
        
    train_losses, test_losses = [], []
    train_accs, test_accs = [], []
    
    for epoch in range(1, epochs + 1):
        model.train()
        t_loss, t_acc = 0.0, 0.0
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            logits = model(X_b)
            loss = loss_fn(logits, y_b)
            preds = torch.round(torch.sigmoid(logits))
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            t_loss += loss.item() * len(X_b)
            t_acc += (accuracy_fn(y_b, preds) / 100.0) * len(X_b)
            
        train_loss = t_loss / len(train_loader.dataset)
        train_acc = (t_acc / len(train_loader.dataset)) * 100.0
        train_losses.append(train_loss)
        train_accs.append(train_acc)
        
        model.eval()
        v_loss, v_acc = 0.0, 0.0
        with torch.inference_mode():
            for X_b, y_b in test_loader:
                X_b, y_b = X_b.to(device), y_b.to(device)
                logits = model(X_b)
                loss = loss_fn(logits, y_b)
                preds = torch.round(torch.sigmoid(logits))
                v_loss += loss.item() * len(X_b)
                v_acc += (accuracy_fn(y_b, preds) / 100.0) * len(X_b)
                
        test_loss = v_loss / len(test_loader.dataset)
        test_acc = (v_acc / len(test_loader.dataset)) * 100.0
        test_losses.append(test_loss)
        test_accs.append(test_acc)
        
        scheduler.step(test_loss)
        if epoch % 5 == 0 or epoch == 1:
            print(f"Epoch: {epoch:02d} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.2f}%")
            
    return train_losses, test_losses, train_accs, test_accs

print("=== Training 1D CNN Model ===")
cnn_tr_loss, cnn_te_loss, cnn_tr_acc, cnn_te_acc = train_and_evaluate(model_cnn, train_loader, test_loader, epochs=EPOCHS)

print("\n=== Training MLP Model ===")
mlp_tr_loss, mlp_te_loss, mlp_tr_acc, mlp_te_acc = train_and_evaluate(model_mlp, train_loader, test_loader, epochs=EPOCHS)

## 6. Confusion Matrices & Heatmaps & Classification Reports

In [6]:
def print_eval(model, name):
    model.eval()
    all_preds, all_targets = [], []
    with torch.inference_mode():
        for X_b, y_b in test_loader:
            X_b = X_b.to(device)
            logits = model(X_b)
            preds = torch.round(torch.sigmoid(logits)).cpu().numpy()
            all_preds.extend(preds)
            all_targets.extend(y_b.numpy())
            
    all_preds = np.array(all_preds).squeeze()
    all_targets = np.array(all_targets).squeeze()
    cm = confusion_matrix(all_targets, all_preds)
    labels = ["Untouch (0)", "Touch (1)"]
    
    # Printed Text Analytics
    print("\n" + "="*50)
    print(f" CONFUSION MATRIX ({name})")
    print("="*50)
    print(cm)
    print("\nClassification Report:\n", classification_report(all_targets, all_preds, target_names=labels))
    print("="*50)
    
    # Visual Heatmap Plot
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels)
    plt.title(f"Confusion Matrix Heatmap - {name}")
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.tight_layout()
    plt.show()

print_eval(model_cnn, "1D CNN")
print_eval(model_mlp, "MLP Network")

## 7. Saving Model Weights

In [7]:
torch.save(model_cnn.state_dict(), "finger_touch_cnn1d.pth")
torch.save(model_mlp.state_dict(), "finger_touch_mlp.pth")
print("Saved model weights to 'finger_touch_cnn1d.pth' and 'finger_touch_mlp.pth'.")